# Camada Bronze: State of Data Brasil

Ingestão das três edições da pesquisa, do CSV de origem para o Data Lake.

## A Bronze é cópia fiel da origem

**Nenhuma modificação, nenhuma correção, nenhuma harmonização.** O arquivo entra
como veio do Kaggle: acento, espaço, maiúscula e nome de coluna original.

Renomear coluna e tirar acento de valor são trabalho da Silver, em
`glue/silver/normalizacao_bronze.py`. A equivalência entre as duas formas está
medida em `docs/VALIDACAO_BRONZE.md` §6.2.


# Importações

In [ ]:
import sys
import os

from pyspark.sql import SparkSession
from pyspark.context import SparkContext
# from awsglue.context import GlueContext
# from awsglue.job import Job
# from awsglue.utils import getResolvedOptions


# Funções

In [ ]:
def ler_df_csv(sessao_spark, caminho):
    """
    Lê o CSV da origem sem inferir nada.

    `inferSchema=false` deixa tudo string de propósito: tipagem é decisão da
    Silver. `multiLine` e `escape` são necessários porque as respostas abertas
    da pesquisa têm quebra de linha e aspas dentro do campo.
    """
    df = (
     sessao_spark.read
     .format("csv")
     .option("header", "true")
     .option("inferSchema", "false")
     .option("multiLine", "true")
     .option("quote", '"')
     .option("escape", '"')
     .option("encoding", "UTF-8")
     .load(caminho)
      )
    return df


def gravar_bronze(df_spark, caminho_saida):
    """
    Grava a Bronze em Parquet, particionado por nada: uma tabela por edição.

    Parquet porque é colunar e o Athena lê direto. O nome das colunas vai como
    veio da origem, inclusive com parêntese e espaço em 2023-2024.
    """
    df_spark.write.mode("overwrite").parquet(caminho_saida)
    print(f"[OK] {caminho_saida}: {df_spark.count()} linhas, {len(df_spark.columns)} colunas")


def exportar_df_para_csv(df_spark, nome_arquivo):
    """Saída em CSV, para inspeção local. Não é o formato da Bronze no S3."""
    df_spark.toPandas().to_csv(nome_arquivo, index=False)


# Iniciando projeto

In [ ]:
sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
sessao = SparkSession.builder.appName("Bronze").getOrCreate()
# job = Job(glueContext)


# Caminhos

No Glue, trocar os caminhos locais pelos do S3. A origem fica em `entrada/` e a
Bronze em `bronze/state_data/`, uma pasta por edição.

In [ ]:
BUCKET = os.getenv("TC3_BUCKET", "s3://lab-000000000000")

# origem, como baixada do Kaggle
caminho_2023 = "/content/State_of_data_BR_2023_Kaggle - df_survey_2023.csv"
caminho_2024 = "/content/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv"
caminho_2025 = "/content/Final Dataset - State of Data 2025-2026 - Kaggle.csv"

# caminho_2023 = f"{BUCKET}/entrada/state_of_data_2023_2024.csv"
# caminho_2024 = f"{BUCKET}/entrada/state_of_data_2024_2025.csv"
# caminho_2025 = f"{BUCKET}/entrada/state_of_data_2025_2026.csv"

PATH_BRONZE = os.getenv("TC3_PATH_BRONZE", f"{BUCKET}/bronze/state_data")


# Bronze 2023-2024

In [ ]:
df_state_data_2023 = ler_df_csv(sessao, caminho_2023)
gravar_bronze(df_state_data_2023, f"{PATH_BRONZE}/bronze_dw_state_data_2023_2024")
# esperado: 5.293 linhas, 399 colunas


# Bronze 2024-2025

In [ ]:
df_state_data_2024 = ler_df_csv(sessao, caminho_2024)
gravar_bronze(df_state_data_2024, f"{PATH_BRONZE}/bronze_dw_state_data_2024_2025")
# esperado: 5.217 linhas, 403 colunas


# Bronze 2025-2026

In [ ]:
df_state_data_2025 = ler_df_csv(sessao, caminho_2025)
gravar_bronze(df_state_data_2025, f"{PATH_BRONZE}/bronze_dw_state_data_2025_2026")
# esperado: 3.495 linhas, 388 colunas


# Conferência do contrato

A Silver recusa a Bronze se o shape não bater. Conferir aqui evita descobrir
lá na frente.

In [ ]:
esperado = {
    "bronze_dw_state_data_2023_2024": (5293, 399),
    "bronze_dw_state_data_2024_2025": (5217, 403),
    "bronze_dw_state_data_2025_2026": (3495, 388),
}
for tabela, (linhas, colunas) in esperado.items():
    df = sessao.read.parquet(f"{PATH_BRONZE}/{tabela}")
    n_l, n_c = df.count(), len(df.columns)
    ok = "OK" if (n_l, n_c) == (linhas, colunas) else "DIVERGENTE"
    print(f"{tabela:34s} {n_l:6d} linhas, {n_c:4d} colunas   {ok}")
